# Spark Ingestion: CSV to Parquet Base

**Goal:** Convert raw CSV to a clean Parquet dataset with a unique ID for tracking enrichment progress.
**Output:** `data/reviews_base/`

In [ ]:
# --- CONFIGURATION ---
import sys, os
from pathlib import Path
# Add src to sys.path
sys.path.append(os.path.abspath("../../../../.."))

from stampli.paths import REVIEWS_CSV, REVIEWS_BASE_PARQUET

CSV_PATH = str(REVIEWS_CSV)
OUTPUT_BASE_PATH = str(REVIEWS_BASE_PARQUET)

In [ ]:
# 1. Setup Spark
try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, expr, monotonically_increasing_id
    from stampli.util.runtime import bootstrap_spark_env
except ImportError:
    print("Please install pyspark: pip install pyspark")

bootstrap_spark_env()

spark = (SparkSession.builder
    .appName("StampliIngestion")
    .config("spark.executor.memory", "16g")
    .config("spark.driver.memory", "16g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created")

In [ ]:
# 2. Read CSV (Latin1 encoding is common for this dataset)
print(f"Reading {CSV_PATH}...")
df = spark.read.option("header", "true") \
    .option("encoding", "ISO-8859-1") \
    .option("inferSchema", "true") \
    .csv(CSV_PATH)

print(f"Raw Count: {df.count()}")
df.printSchema()

In [ ]:
# 3. Add Unique ID
# We utilize monotonically_increasing_id() for a fast, unique 64-bit integer ID.
# Alternatively, we could generate a UUID string if global uniqueness across clusters is needed.
df_with_id = df.withColumn("review_uid", expr("uuid()"))

# 4. Write to Parquet (Overwrite Initial Base)
print(f"Writing to {OUTPUT_BASE_PATH}...")
df_with_id.write.mode("overwrite").parquet(OUTPUT_BASE_PATH)

print("Done.")